In [1]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [2]:
!pip install xgboost

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

train.head()

(1352, 51)
(339, 50)


,CoilID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X41,X42,X43,X44,X45,X46,X47,X48,X49,Y
0,487,854.787195,501.088868,414.841484,710.583316,662.072013,656.076977,547.040479,563.653582,495.296785,...,0.201645,0.047960,0.267467,0.052247,-0.893174,0.000000,0.028925,0.000534,0.010797,0.0
1,44,1056.526699,868.083321,622.879982,725.276469,665.235554,647.450550,552.333202,565.105074,493.310075,...,0.644403,0.000000,0.341870,0.153513,25.471899,0.002520,0.033281,0.028349,0.079602,0.0
2,192,1095.648362,668.112517,695.787904,716.773671,662.843475,657.542380,549.863867,546.210823,482.814753,...,0.486502,0.000000,0.202539,0.168192,-25.764196,0.002072,0.033878,0.000000,0.058266,0.0
3,1552,1050.943543,660.340015,440.280245,611.562496,628.081103,561.397721,456.816210,550.103433,378.353283,...,1.198010,0.020787,0.288786,0.329108,1.033840,0.000250,0.045490,0.039004,0.004850,0.0
4,1190,1091.640314,297.363775,842.665620,749.160886,652.992309,615.576656,608.364764,549.756758,487.753140,...,0.237231,0.000841,0.257281,0.112637,-11.130157,0.002376,0.031298,0.003623,0.018434,0.0


In [5]:
X = train.drop(["Y", "CoilID"], axis=1)
y = train["Y"]

X_test = test.drop(["CoilID"], axis=1)

print(X.shape)
print(y.shape)

(1352, 49)
(1352,)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

(1081, 49)
(271, 49)


In [7]:
print(y.value_counts())
print()
print(y.value_counts(normalize=True))

Y
0.0    1286
1.0      66
Name: count, dtype: int64

Y
0.0    0.951183
1.0    0.048817
Name: proportion, dtype: float64


In [8]:
negative = 1286
positive = 66

scale_pos_weight = negative / positive

print(scale_pos_weight)

19.484848484848484


In [9]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=19.48,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [10]:
probs = model.predict_proba(X_val)[:, 1]

print(probs[:10])

[0.00012224 0.00031491 0.00028415 0.00228117 0.00243345 0.01471032
 0.00020824 0.00066503 0.00095858 0.00034408]


In [11]:
best_threshold = 0
best_precision = 0

for t in [i/100 for i in range(1,100)]:

    preds = (probs >= t).astype(int)

    recall = recall_score(y_val, preds)
    precision = precision_score(y_val, preds, zero_division=0)

    if recall == 1.0:
        print(f"Threshold: {t:.2f}")
        print(f"Recall: {recall:.2f}")
        print(f"Precision: {precision:.2f}")
        print("-------------------")

        if precision > best_precision:
            best_precision = precision
            best_threshold = t

print("Best Threshold:", best_threshold)
print("Best Precision:", best_precision)

Best Threshold: 0
Best Precision: 0


In [14]:
threshold = 0.01

preds = (probs >= threshold).astype(int)

print("Recall:", recall_score(y_val, preds))
print("Precision:", precision_score(y_val, preds))
print()

print(confusion_matrix(y_val, preds))

Recall: 0.6153846153846154
Precision: 0.1509433962264151

[[213  45]
 [  5   8]]


In [15]:
model = XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    scale_pos_weight=30,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [16]:
probs = model.predict_proba(X_val)[:, 1]

print(probs[:10])

[1.5662218e-04 2.9497821e-04 6.6350498e-05 2.4553312e-03 4.6133590e-03
 3.8949412e-03 1.8585839e-04 6.0154981e-04 1.5970180e-04 1.4172628e-04]


In [17]:
for t in [0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05]:

    preds = (probs >= t).astype(int)

    recall = recall_score(y_val, preds)
    precision = precision_score(y_val, preds, zero_division=0)

    print(f"Threshold = {t}")
    print("Recall =", recall)
    print("Precision =", precision)
    print(confusion_matrix(y_val, preds))
    print("-------------------")

Threshold = 0.001
Recall = 0.7692307692307693
Precision = 0.09259259259259259
[[160  98]
 [  3  10]]
-------------------
Threshold = 0.003
Recall = 0.6923076923076923
Precision = 0.125
[[195  63]
 [  4   9]]
-------------------
Threshold = 0.005
Recall = 0.6153846153846154
Precision = 0.14545454545454545
[[211  47]
 [  5   8]]
-------------------
Threshold = 0.01
Recall = 0.5384615384615384
Precision = 0.16279069767441862
[[222  36]
 [  6   7]]
-------------------
Threshold = 0.02
Recall = 0.5384615384615384
Precision = 0.20588235294117646
[[231  27]
 [  6   7]]
-------------------
Threshold = 0.03
Recall = 0.5384615384615384
Precision = 0.22580645161290322
[[234  24]
 [  6   7]]
-------------------
Threshold = 0.05
Recall = 0.38461538461538464
Precision = 0.18518518518518517
[[236  22]
 [  8   5]]
-------------------


In [18]:
!pip install imbalanced-learn

In [19]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print(X_train_imputed.shape)

(1081, 49)


In [20]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_imputed,
    y_train
)

print(X_train_smote.shape)
print(y_train_smote.value_counts())

(2056, 49)
Y
0.0    1028
1.0    1028
Name: count, dtype: int64


In [21]:
model = XGBClassifier(
    n_estimators=1200,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1200,
              n_jobs=None, num_parallel_tree=None, ...)

In [22]:
probs = model.predict_proba(X_val_imputed)[:, 1]

for t in [0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05]:

    preds = (probs >= t).astype(int)

    recall = recall_score(y_val, preds)
    precision = precision_score(y_val, preds, zero_division=0)

    print(f"Threshold = {t}")
    print("Recall =", recall)
    print("Precision =", precision)
    print(confusion_matrix(y_val, preds))
    print("-------------------")

Threshold = 0.001
Recall = 0.9230769230769231
Precision = 0.10619469026548672
[[157 101]
 [  1  12]]
-------------------
Threshold = 0.003
Recall = 0.6923076923076923
Precision = 0.10588235294117647
[[182  76]
 [  4   9]]
-------------------
Threshold = 0.005
Recall = 0.6153846153846154
Precision = 0.11764705882352941
[[198  60]
 [  5   8]]
-------------------
Threshold = 0.01
Recall = 0.5384615384615384
Precision = 0.125
[[209  49]
 [  6   7]]
-------------------
Threshold = 0.02
Recall = 0.5384615384615384
Precision = 0.14
[[215  43]
 [  6   7]]
-------------------
Threshold = 0.03
Recall = 0.5384615384615384
Precision = 0.16666666666666666
[[223  35]
 [  6   7]]
-------------------
Threshold = 0.05
Recall = 0.5384615384615384
Precision = 0.21212121212121213
[[232  26]
 [  6   7]]
-------------------


In [24]:
test_probs = model.predict_proba(X_test_imputed)[:, 1]

threshold = 0.001

test_preds = (test_probs >= threshold).astype(int)

print(test_preds[:20])

[1 0 1 0 1 0 1 0 0 0 0 1 0 0 0 1 0 0 1 1]


In [25]:
submission = pd.DataFrame({
    "CoilID": test["CoilID"],
    "Y": test_preds
})

submission.head()

,CoilID,Y
0,711,1
1,1542,0
2,1232,1
3,600,0
4,1087,1


In [26]:
submission.to_csv("expected_submission.csv", index=False)

print("Submission file created successfully!")

Submission file created successfully!


In [27]:
from google.colab import files

files.download("expected_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>